# TVM Optimized Tomography

## Useful Defines

In [12]:
import numpy as np

# -------------------- frequency utilities --------------------

def _fftfreq_radius(shape, voxel_size=1.0):
    freqs = [np.fft.fftfreq(n, d=voxel_size) for n in shape]
    grids = np.meshgrid(*freqs, indexing="ij")
    r = np.sqrt(sum(g**2 for g in grids))
    return r

# -------------------- filtering & masks --------------------

def _lowpass_filter(arr, voxel_size=1.0, cutoff_cyc_per_unit=0.25, trans_width=0.0):
    """
    3D isotropic low-pass filter.
    cutoff_cyc_per_unit: frequency cutoff (cycles / unit)
    trans_width: optional cosine transition width for a soft edge
    """
    F = np.fft.fftn(arr)
    r = _fftfreq_radius(arr.shape, voxel_size)
    mask = np.zeros_like(r, dtype=np.float32)
    if trans_width > 0:
        inner = r <= (cutoff_cyc_per_unit - trans_width)
        outer = r >= (cutoff_cyc_per_unit + trans_width)
        mid = (~inner) & (~outer)
        mask[inner] = 1.0
        mask[mid] = 0.5 * (1 + np.cos(np.pi * (r[mid] - (cutoff_cyc_per_unit - trans_width)) / (2*trans_width)))
    else:
        mask[r <= cutoff_cyc_per_unit] = 1.0
    F *= mask
    return np.fft.ifftn(F).real

def _checkerboard_masks(shape):
    """Return boolean masks for even/odd checkerboard pattern in 3D."""
    idx = np.meshgrid(*[np.arange(n) for n in shape], indexing="ij")
    parity = sum(idx) % 2
    m0 = (parity == 0)
    m1 = ~m0
    return m0, m1

# -------------------- FSC core --------------------

def _fsc(vol1, vol2, voxel_size=1.0, df=0.01, eps=1e-12):
    """Compute Fourier shell correlation between two 3D arrays."""
    assert vol1.shape == vol2.shape
    vol1 = vol1 - np.mean(vol1)
    vol2 = vol2 - np.mean(vol2)

    F1 = np.fft.fftn(vol1)
    F2 = np.fft.fftn(vol2)
    r = _fftfreq_radius(vol1.shape, voxel_size)

    r_max = r.max()
    edges = np.arange(0, r_max + df, df)
    bins = np.digitize(r.ravel(), edges) - 1
    n_shells = edges.size - 1

    cross = (F1 * np.conj(F2)).ravel().real
    p1 = (np.abs(F1)**2).ravel()
    p2 = (np.abs(F2)**2).ravel()

    valid = (bins >= 0) & (bins < n_shells)
    b = bins[valid]
    num = np.bincount(b, weights=cross[valid], minlength=n_shells)
    den1 = np.bincount(b, weights=p1[valid], minlength=n_shells)
    den2 = np.bincount(b, weights=p2[valid], minlength=n_shells)

    fsc = num / (np.sqrt(den1 * den2) + eps)
    freqs = 0.5 * (edges[:-1] + edges[1:])
    return freqs, fsc

# -------------------- Checkerboard SFSC --------------------

def sfsc_checkerboard(volume, voxel_size=1.0, df=0.01, trans_width=0.0):
    """
    Self-Fourier Shell Correlation (3D checkerboard version)
    Steps:
      1) Low-pass to 0.25 / voxel_size (to avoid aliasing)
      2) Split volume into two interleaved sublattices (x+y+z mod 2)
      3) Reconstruct both halves by low-pass interpolation
      4) Compute FSC between the two reconstructed maps
    """
    vol = volume.astype(np.float32)
    cutoff = 0.25 / voxel_size

    # Step 1: anti-alias filter
    vol_lp = _lowpass_filter(vol, voxel_size, cutoff, trans_width)
    #vol_lp = vol
    # Step 2: checkerboard split
    m0, m1 = _checkerboard_masks(vol.shape)
    v0_sparse = np.zeros_like(vol_lp)
    v1_sparse = np.zeros_like(vol_lp)
    v0_sparse[m0] = vol_lp[m0]
    v1_sparse[m1] = vol_lp[m1]

    # Step 3: reconstruct each half to full grid
    v0_rec = _lowpass_filter(v0_sparse, voxel_size, cutoff, trans_width)
    v1_rec = _lowpass_filter(v1_sparse, voxel_size, cutoff, trans_width)

    # Step 4: FSC between reconstructed halves
    freqs, fsc = _fsc(v0_rec, v1_rec, voxel_size, df=df)
    resolution = fsc_resolution(freqs, fsc, threshold=0.143)
    if resolution is None:
        resolution = voxel_size
    return resolution, freqs, fsc, v0_rec, v1_rec

# -------------------- Resolution utility --------------------

def fsc_resolution(freqs, fsc, threshold=0.143):
    """Linear interpolation to find 1/f at threshold crossing."""
    below = np.where(fsc < threshold)[0]
    if not len(below):
        return None
    i = below[0]
    if i == 0:
        f_cut = freqs[i]
    else:
        f1, f2 = freqs[i-1], freqs[i]
        y1, y2 = fsc[i-1], fsc[i]
        f_cut = f1 + (threshold - y1) * (f2 - f1) / (y2 - y1)
    return 1.0 / f_cut



In [2]:
import tomosipo as ts
import numpy as np
from tomobase.data import Volume, Sinogram, volume
import cupy as cp
import os
def project(volume, angles):
    """
    Project a 3D volume to generate sinograms at specified angles.

    Parameters:
    - volume: 3D numpy array of shape (num_slices, height, width)
    - angles: 1D numpy array of projection angles in radians

    Returns:
    - sinograms: 3D numpy array of shape (num_angles, num_detectors, num_slices)
    """
    volume.data = volume.data.transpose(2,0,1)  # (S,H,W)->(W,S,H) 021
    volume.data = cp.asarray(volume.data)

    angles_rad = np.radians(angles+90)
    vg = ts.volume(shape=(128,128,128))
    pg = ts.parallel(angles=angles_rad, shape=(128, 128))
    A = ts.operator(vg, pg)


    sinogram = cp.asnumpy(A(volume.data).transpose(1,0, 2))
    return Sinogram(sinogram, angles)

def reconstruct_tvm(sinogram, vol_file=None, num_iterations=100, lambda_tv=0.1 ):
    """
    Reconstruct a 3D volume using SIRT with Total Variation regularization.
    
    Parameters:
    - sinogram: Sinogram object with projection data
    - num_iterations: Number of iterations
    - lambda_tv: TV regularization strength (higher = more smoothing)
    
    Returns:
    - volume: Reconstructed Volume object
    """
    angles_rad = np.radians(sinogram.angles + 90)
    vg = ts.volume(shape=(sinogram.data.shape[1],sinogram.data.shape[1], sinogram.data.shape[2]), size=(1, 1, 1))
    pg = ts.parallel(angles=angles_rad, shape=(sinogram.data.shape[1], sinogram.data.shape[2]), size=(1.0, 1.0))
    
    A = ts.operator(vg, pg)
    
    # SIRT weights
    R =  1 / A(np.ones(A.domain_shape))
    R = np.clip(R, a_min=None, a_max=1 / ts.epsilon)
    R = cp.asarray(R)
    C = 1 / A.T(np.ones(A.range_shape))
    C = np.clip(C, a_min=None, a_max=1 / ts.epsilon)
    C = cp.asarray(C)
    
    y = cp.asarray(sinogram.data.transpose(1, 0, 2))
    if vol_file is None:
        x_rec = cp.asarray(np.zeros(A.domain_shape, dtype=np.float32))
    else:
        x_rec = cp.asarray(Volume.from_file(vol_file).data)
        num_iterations =  int(vol_file.name.split('_')[0]) 
    
    for i in range(num_iterations):
        # Standard SIRT update
        residual = y - A(x_rec)
        sirt_update = C * A.T(R * residual)
        
        # TV gradient (using finite differences)
        tv_grad = compute_tv_gradient(x_rec)
        
        # Combined update with TV regularization
        x_rec += sirt_update - lambda_tv * tv_grad
        #x_rec += sirt_update
        
        # Non-negativity constraint (optional but common in tomography)
        x_rec = cp.maximum(x_rec, 0)
    
    x_rec = cp.asnumpy(x_rec)
    x_rec = x_rec.transpose(1, 2, 0)  # (W,S,H)->(S,H,W) 120
    return Volume(x_rec)


def compute_tv_gradient(volume):
    """
    Compute the gradient of the Total Variation functional.
    Uses isotropic TV: sum of gradient magnitudes.
    """
    # Compute gradients along each axis using finite differences
    grad_x = cp.zeros_like(volume)
    grad_y = cp.zeros_like(volume)
    grad_z = cp.zeros_like(volume)
    
    # Forward differences
    grad_x[:-1, :, :] = volume[1:, :, :] - volume[:-1, :, :]
    grad_y[:, :-1, :] = volume[:, 1:, :] - volume[:, :-1, :]
    grad_z[:, :, :-1] = volume[:, :, 1:] - volume[:, :, :-1]
    
    # Gradient magnitude (with small epsilon for numerical stability)
    eps = 1e-8
    grad_mag = cp.sqrt(grad_x**2 + grad_y**2 + grad_z**2 + eps)
    
    # Compute TV gradient (divergence of normalized gradient)
    tv_grad = cp.zeros_like(volume)
    
    # Backward divergence for x
    div_x = cp.zeros_like(volume)
    div_x[1:-1, :, :] = (grad_x[1:-1, :, :] / grad_mag[1:-1, :, :] - 
                          grad_x[:-2, :, :] / grad_mag[:-2, :, :])
    div_x[0, :, :] = grad_x[0, :, :] / grad_mag[0, :, :]
    div_x[-1, :, :] = -grad_x[-2, :, :] / grad_mag[-2, :, :]
    
    # Backward divergence for y
    div_y = cp.zeros_like(volume)
    div_y[:, 1:-1, :] = (grad_y[:, 1:-1, :] / grad_mag[:, 1:-1, :] - 
                          grad_y[:, :-2, :] / grad_mag[:, :-2, :])
    div_y[:, 0, :] = grad_y[:, 0, :] / grad_mag[:, 0, :]
    div_y[:, -1, :] = -grad_y[:, -2, :] / grad_mag[:, -2, :]
    
    # Backward divergence for z
    div_z = cp.zeros_like(volume)
    div_z[:, :, 1:-1] = (grad_z[:, :, 1:-1] / grad_mag[:, :, 1:-1] - 
                          grad_z[:, :, :-2] / grad_mag[:, :, :-2])
    div_z[:, :, 0] = grad_z[:, :, 0] / grad_mag[:, :, 0]
    div_z[:, :, -1] = -grad_z[:, :, -2] / grad_mag[:, :, -2]
    
    tv_grad = -(div_x + div_y + div_z)
    
    return tv_grad

ModuleNotFoundError: No module named 'tomobase.data.image'

In [11]:
import copy 
from itertools import combinations
import copy
import os 
import gc  # Garbage collector for memory cleanup
from pathlib import Path



def find_and_reconstruct(folder,  start, end, sino, recon, *args, **kwargs):
    matches = list(Path(folder).glob(f"*_recon_{start}_{end}.rec"))
    if matches:
        vol = recon(sino, vol_file=matches[0], *args, **kwargs)
    else:
        vol = recon(sino, vol_file=None, *args, **kwargs)
    return vol

def subsection_sinogram(sino, start, end):
    sinogram = copy.deepcopy(sino)
    sinogram.data = sinogram.data[start:end,:, :]
    sinogram.angles = sinogram.angles[start:end]
    sinogram.times = sinogram.times[start:end]
    return sinogram

def sfsc_split_projection(sinogram, recon_func, folder='', start=0, end=0, voxel_size=1.0, df=0.01, **recon_kwargs):
    """
    Self-FSC using split-projection method.
    Proper for limited-angle tomography where checkerboard fails.
    
    Randomly splits projections into two halves, reconstructs each,
    then computes FSC between the two independent reconstructions.
    """
    matches = list(Path(folder).glob(f"*_recon_{start}_{end}.rec"))
    half = sinogram.data.shape[0] // 2

    # Create two sub-sinograms
    sino1 = subsection_sinogram(sinogram, 0, half)
    sino2 = subsection_sinogram(sinogram, half, sinogram.data.shape[0])

    vol1 = recon_func(sino1, vol_file=matches[0] if matches else None, **recon_kwargs)
    vol2 = recon_func(sino2, vol_file=matches[0] if matches else None, **recon_kwargs)
    
    # Compute FSC between the two independent reconstructions
    freqs, fsc = _fsc(vol1.data, vol2.data, voxel_size, df=df)
    resolution = fsc_resolution(freqs, fsc, threshold=0.143)
    
    if resolution is None:
        # If FSC never drops, return Nyquist limit
        resolution = 2 * voxel_size
    
    # Clean up large temporary volumes
    del vol1, vol2, sino1, sino2
    gc.collect()
    
    return resolution, freqs, fsc

## Generalized Process

In [ ]:


def get_windows(sino, recon, windows=None, folder='', *args, **kwargs):
    tolerance = 0.05
    rolling_ave = 3
    if windows == None:
        windows = np.zeros((len(sino.times), 3), int)
    
        for i in range(len(sino.times)):
            windows[i,0] = i
            windows[i,1] = i+20
            windows[i,2] = 0

    tolerance = sino.pixelsize    
    for i in range(len(sino.times)):
        if windows[i,1]>len(sino.times):
            for j in range(i, windows.shape[0]):
                windows[j,2] = 0
            break

        break_out = False
        while not break_out:
            sino_subsect = subsection_sinogram(sino, windows[i, 0], windows[i, 1])
            
            # Use split-projection FSC
            resolution = sfsc_split_projection(sino_subsect, recon, folder=folder, start=windows[i, 0], end=windows[i, 1], voxel_size=sino.pixelsize,
                                              num_iterations=kwargs.get('num_iterations', 100),
                                              lambda_tv=kwargs.get('lambda_tv', 0.1))[0]

            if windows[i,1]>windows[i,0]+rolling_ave:
                rolling_decs = np.zeros(rolling_ave)
                for k in range(1, rolling_ave+1):
                    sino_subsect_dec = subsection_sinogram(sino, windows[i, 0], windows[i, 1]-k)
                    rolling_decs[k-1] = sfsc_split_projection(sino_subsect_dec, recon, folder=folder, start=windows[i, 0], end=windows[i, 1]-k, voxel_size=sino.pixelsize,
                                                      num_iterations=kwargs.get('num_iterations', 100),
                                                      lambda_tv=kwargs.get('lambda_tv', 0.1))[0]
                    # Cleanup
                    del sino_subsect_dec
                resolution_dec = np.mean(rolling_decs)
                resolution_dec_std = np.std(rolling_decs) if rolling_ave > 1 else 0
                del rolling_decs
            else:
                resolution_dec = sino.pixelsize*sino.data.shape[1]
                resolution_dec_std = 0

            if windows[i,1]+rolling_ave<len(sino.times):
                rolling_incs = np.zeros(rolling_ave)
                for k in range(1, rolling_ave+1):
                    sino_subsect_inc = subsection_sinogram(sino, windows[i, 0], windows[i, 1]+k)
                    rolling_incs[k-1] = sfsc_split_projection(sino_subsect_inc, recon, folder=folder, start=windows[i, 0], end=windows[i, 1]+k, voxel_size=sino.pixelsize,
                                                      num_iterations=kwargs.get('num_iterations', 100),
                                                      lambda_tv=kwargs.get('lambda_tv', 0.1))[0]
                    # Cleanup
                    del sino_subsect_inc
                resolution_inc = np.mean(rolling_incs)
                resolution_inc_std = np.std(rolling_incs) if rolling_ave > 1 else 0
                del rolling_incs
            else:
                resolution_inc = sino.pixelsize*sino.data.shape[1]
                resolution_inc_std = 0
            
            print("--------------------------------------------------")
            print(f"Iterations: {kwargs.get('num_iterations', 100)} | Lambda TV: {kwargs.get('lambda_tv', 0.1)}")
            print(f"Window {i}: {windows[i,0]}-{windows[i,1]} | Res: {resolution:.3f}")
            print(f"  Dec: {resolution_dec:.3f}±{resolution_dec_std:.3f} | Inc: {resolution_inc:.3f}±{resolution_inc_std:.3f}")
            
            if resolution_dec + (resolution*tolerance) + resolution_dec_std < resolution:
                windows[i,1] -= 1
                print(f"  → Decreasing window to {windows[i,1]}")
            elif resolution_inc + (resolution*tolerance) + resolution_inc_std < resolution:
                windows[i,1] += 1
                print(f"  → Increasing window to {windows[i,1]}")
            else:
                windows[i,2] = 1
                break_out = True
                print(f"  → Window optimized!")
            
            # Cleanup iteration
            del sino_subsect
            gc.collect()

    return windows

def dynamic_reconstruction(sino, folder, iterations=100, lambda_tv=0.1): 
    for iter_idx in range(iterations):
        vol = reconstruct_tvm(sino, num_iterations=100, lambda_tv=lambda_tv)
        
        # Use split-projection FSC
        full_resolution = sfsc_split_projection(sino, reconstruct_tvm, folder='', start=0, end=len(sino.times), voxel_size=sino.pixelsize,
                                               num_iterations=100, lambda_tv=lambda_tv)[0]
        print(f"\n{'='*60}")
        print(f"TVM Iteration {iter_idx+1}/{iterations} | Full Volume Reconstruction | Res: {full_resolution:.3f}")
        print(f"{'='*60}\n")

        if iter_idx == 0:
            win = get_windows(sino, reconstruct_tvm, windows=None, folder=folder, num_iterations=iter_idx+1, lambda_tv=lambda_tv)
        else:
            win = get_windows(sino, reconstruct_tvm, windows=win, folder=folder, num_iterations=iter_idx+1, lambda_tv=lambda_tv)

        for win_idx in range(win.shape[0]):
            if win[win_idx,2]==1:
                sino_subset = subsection_sinogram(sino, win[win_idx,0], win[win_idx,1])
                recon = find_and_reconstruct(folder, win[win_idx,0], win[win_idx,1], sino_subset, reconstruct_tvm, num_iterations=iterations, lambda_tv=lambda_tv)
                filename = f"{folder}/{iter_idx}_recon_{win[win_idx,0]}_{win[win_idx,1]}.rec"
                matches = list(Path(folder).glob(f"*_recon_{win[win_idx,0]}_{win[win_idx,1]}.rec"))
                for match in matches:
                    os.remove(match)
                recon.to_file(filename)
                print(f"Saved: {filename}")
                
                # Cleanup
                del sino_subset, recon
        
        # Cleanup iteration
        del vol
        gc.collect()

    # Cleanup old files
    for file in os.listdir(folder):
        iteration_file = int(file.split('_')[0])
        if iteration_file < iterations: 
            path = os.path.join(folder, file)
            if os.path.exists(path):
                os.remove(path)

In [14]:
folder_name= r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\Volumes128\Rod-D-4.0'
import tomobase.processes as processes
from tomobase.tiltschemes import GRS

'''
tiltscheme = GRS(-70, 70,0)
angles = np.array([tiltscheme.get_angle() for i in range(100)])

for i in range( 1,101):
    vol_path = os.path.join(folder_name, f'{i}data.rec')
    vol = Volume.from_file(vol_path)
    sino_img = processes.project(vol, [angles[i-1]])
    sino_img.times = np.array([i])
    if i == 1:
        sino = copy.deepcopy(sino_img)
    else:
        sino.data = np.concatenate((sino.data, sino_img.data), axis=0)
        sino.angles = np.concatenate((sino.angles, sino_img.angles), axis=0)
        sino.times = np.concatenate((sino.times, sino_img.times), axis=0)
'''
folder_name = r'D:\dynamic_sims\rod'
dynamic_reconstruction(sino, folder_name, iterations=100, lambda_tv=0.1)

c:\Users\TCraig\AppData\Local\miniconda3\envs\total-env\Lib\site-packages\tomosipo\links\numpy.py:27: UserWarning: The parameter initial_value is of type float64; expected `np.float32`. The type has been Automatically converted. Use `ts.link(x.astype(np.float32))' to inhibit this warning. 
  warnings.warn(



TVM Iteration 1/100 | Full Volume Reconstruction | Res: 1.379

--------------------------------------------------
Iterations: 1 | Lambda TV: 0.1


NameError: name 'adjustment_count' is not defined

## Memory Management

Run this cell if you need to manually clear variables and free memory:

In [6]:
# Manual memory cleanup - run this cell when needed
import gc

# Option 1: Delete specific variables
# del vol, sino, resolution  # Uncomment and list variables to delete

# Option 2: Force garbage collection
gc.collect()
print("Garbage collection complete")

# Option 3: See what's taking up memory
import sys

# Get top 10 largest objects in memory
def get_size(obj):
    try:
        return sys.getsizeof(obj)
    except:
        return 0

# Show current namespace variables (comment out if slow)
# local_vars = list(locals().items())
# local_vars.sort(key=lambda x: get_size(x[1]), reverse=True)
# for name, obj in local_vars[:10]:
#     print(f"{name}: {get_size(obj) / 1024 / 1024:.2f} MB")

# Option 4: Clear everything except imports (CAREFUL!)
# %reset -f  # This clears ALL variables!

Garbage collection complete
